# TinyLlama Roleplay Fine-Tuning (Colab Ready)

This notebook trains **TinyLlama** for roleplay/chat behavior using:
- OpenAssistant (`OpenAssistant/oasst1`)
- ShareGPT (`anon8231489123/ShareGPT_Vicuna_unfiltered`)
- CharacterAI (`ehartford/characterai`)

It uses **LoRA + QLoRA**, gradient checkpointing, and mixed precision.


## 1) Install dependencies


In [ ]:
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes sentencepiece


## 2) Imports and global configuration


In [ ]:
import random
import torch
from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = "./tinyllama-roleplay-lora"
MAX_SEQ_LEN = 1024
MAX_SAMPLES_PER_DATASET = 30000


## 3) Load TinyLlama (QLoRA setup)


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()


## 4) Load datasets


In [ ]:
oasst = load_dataset("OpenAssistant/oasst1", split="train")
sharegpt = load_dataset("anon8231489123/ShareGPT_Vicuna_unfiltered", split="train")
characterai = load_dataset("ehartford/characterai", split="train")

print(oasst)
print(sharegpt)
print(characterai)


## 5) Clean datasets


In [ ]:
def pick_first(dct, keys, default=""):
    for key in keys:
        if isinstance(dct, dict) and key in dct and dct[key] is not None:
            return dct[key]
    return default

def format_turns_as_chat(turns):
    lines = []
    for turn in turns:
        if not isinstance(turn, dict):
            continue
        role = str(turn.get("from", turn.get("role", "user"))).lower()
        value = turn.get("value", turn.get("text", ""))
        if not value:
            continue
        if role in ["human", "user", "prompter"]:
            prefix = "User"
        elif role in ["gpt", "assistant"]:
            prefix = "Assistant"
        else:
            prefix = role.capitalize()
        lines.append(f"{prefix}: {value}".strip())
    return "\n".join(lines).strip()

def map_oasst(example):
    role = str(example.get("role", "")).lower()
    content = (example.get("text", "") or "").strip()
    if role == "assistant":
        text = f"Assistant: {content}"
    elif role == "prompter":
        text = f"User: {content}"
    else:
        text = content
    return {"text": text.strip()}

def map_sharegpt(example):
    conv = example.get("conversations", None)
    if isinstance(conv, list):
        text = format_turns_as_chat(conv)
    else:
        q = pick_first(example, ["instruction", "prompt", "question"])
        a = pick_first(example, ["output", "response", "answer"])
        text = f"User: {q}\nAssistant: {a}"
    return {"text": text.strip()}

def map_characterai(example):
    if "conversations" in example and isinstance(example["conversations"], list):
        text = format_turns_as_chat(example["conversations"])
    else:
        user_msg = pick_first(example, ["src", "input", "prompt", "question", "human"])
        bot_msg = pick_first(example, ["tgt", "output", "response", "answer", "assistant"])
        text = f"User: {user_msg}\nAssistant: {bot_msg}"
    return {"text": text.strip()}

def clean_text(example):
    text = (example.get("text", "") or "").strip()
    return {"text": text if len(text) >= 20 else ""}

oasst_text = oasst.map(map_oasst, remove_columns=oasst.column_names)
sharegpt_text = sharegpt.map(map_sharegpt, remove_columns=sharegpt.column_names)
characterai_text = characterai.map(map_characterai, remove_columns=characterai.column_names)

def postprocess(ds):
    ds = ds.map(clean_text)
    ds = ds.filter(lambda x: len(x["text"]) > 0)
    if len(ds) > MAX_SAMPLES_PER_DATASET:
        ds = ds.shuffle(seed=SEED).select(range(MAX_SAMPLES_PER_DATASET))
    return ds

oasst_text = postprocess(oasst_text)
sharegpt_text = postprocess(sharegpt_text)
characterai_text = postprocess(characterai_text)

train_dataset = concatenate_datasets([oasst_text, sharegpt_text, characterai_text]).shuffle(seed=SEED)
print(train_dataset)
print(train_dataset[0]["text"][:500])


## 6) Tokenize dataset


In [ ]:
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,
    )

tokenized_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing",
).train_test_split(test_size=0.02, seed=SEED)

print(tokenized_dataset)


## 7) Configure LoRA


In [ ]:
peft_config = LoraConfig(
    r=64,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)


## 8) Train model


In [ ]:
bf16_supported = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    weight_decay=0.01,
    logging_steps=20,
    evaluation_strategy="steps",
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    fp16=not bf16_supported,
    bf16=bf16_supported,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to="none",
    seed=SEED,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    peft_config=peft_config,
    data_collator=data_collator,
)

trainer.train()


## 9) Evaluate model


In [ ]:
metrics = trainer.evaluate()
print(metrics)


In [ ]:
prompt = "User: I need help creating a fantasy tavern keeper character profile.\nAssistant:"
inputs = tokenizer(prompt, return_tensors="pt").to(trainer.model.device)

with torch.no_grad():
    outputs = trainer.model.generate(
        **inputs,
        max_new_tokens=180,
        temperature=0.8,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


## 10) Export model


In [ ]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved LoRA adapter and tokenizer to: {OUTPUT_DIR}")
